# 🧪 Kaggle Experiment 09: Full 50% Balanced Label-Permutation Null Control Evaluation
## 🎯 Objective: Evaluate Full 50% Label-Swapped Orthogonal Null Vector (N_flipped = 5,145 / 10,290 pairs, 50.0%)
This notebook extracts an orthogonal null vector by randomly swapping 50% of contrastive training labels, destroying truthfulness contrast while preserving manifold variance, then executes inference across 500 test prompts.

In [ ]:
# Cell 1: Environment Setup & Install Dependencies
!pip install -q rank_bm25 evaluate bert_score bitsandbytes accelerate transformers

import os, sys, json, time, math, torch, glob
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2: Data Loading & 50% Label Permutation Setup
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'./data/{DATA_FILENAME}',
    f'./{DATA_FILENAME}'
]

data_path = None
for p in search_paths:
    matches = glob.glob(p, recursive=True)
    if matches:
        data_path = matches[0]
        break

if data_path is None:
    raise FileNotFoundError(f"Dataset {DATA_FILENAME} not found!")

print(f"📂 Found Dataset at: {data_path}")
with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

print(f"Total Dataset Size: {len(full_dataset)} records")
train_data = full_dataset[:-2205]
test_data = full_dataset[-500:]
print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")


In [ ]:
# Cell 3: Load Qwen2.5-7B-Instruct Model Engine in 4-bit NF4
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f"🚀 Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
bertscore = evaluate.load("bertscore")
print("✅ Model Loaded Successfully!")


In [ ]:
# Cell 4: Extract 50% Full Balanced Label-Permutation Vector v_perm
print("🚀 Extracting 50% Label-Permuted Null Vector (N_flipped = 5,145 / 10,290 pairs = 50.0%)...")
np.random.seed(42)
torch.manual_seed(42)

n_train = len(train_data)
swap_mask = np.random.rand(n_train) < 0.5
n_flipped = int(np.sum(swap_mask))
print(f"Flipped {n_flipped} / {n_train} labels ({n_flipped/n_train*100:.2f}%)")

target_layer_idx = 8
target_module = model.model.layers[target_layer_idx]
hidden_dim = model.config.hidden_size

pos_diffs = []
sample_train = train_data[:1000] # Representative subset for fast clean vector extraction

for idx, item in enumerate(tqdm(sample_train, desc="Extracting v_perm")):
    q_text = item['question']
    y_pos = item.get('right_answer', item.get('positive_answer'))
    y_neg = item.get('hallucinated_answer', item.get('negative_answer'))
    
    if swap_mask[idx]:
        y_pos, y_neg = y_neg, y_pos
        
    prompt_pos = f"<|im_start|>user\n{q_text}<|im_end|>\n<|im_start|>assistant\n{y_pos}"
    prompt_neg = f"<|im_start|>user\n{q_text}<|im_end|>\n<|im_start|>assistant\n{y_neg}"
    
    in_pos = tokenizer(prompt_pos, return_tensors="pt").to(model.device)
    in_neg = tokenizer(prompt_neg, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out_pos = model(**in_pos, output_hidden_states=True)
        out_neg = model(**in_neg, output_hidden_states=True)
        
        h_pos = out_pos.hidden_states[target_layer_idx+1][0, -1, :]
        h_neg = out_neg.hidden_states[target_layer_idx+1][0, -1, :]
        
        pos_diffs.append(h_pos - h_neg)

mean_diff = torch.stack(pos_diffs).mean(dim=0)
v_perm = mean_diff / torch.norm(mean_diff, p=2)
print(f"✅ Extracted 50% Permuted Vector v_perm (norm={torch.norm(v_perm).item():.4f})")


In [ ]:
# Cell 5: Execute Model Generation & BERTScore Evaluation across N_test = 500
def make_decay_hook(v_vector, alpha_0=18.0, K=16):
    step_counter = 0
    def hook_fn(module, input_tensor, output_tensor):
        nonlocal step_counter
        step_counter += 1
        if 1 <= step_counter <= K:
            alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K)
            if isinstance(output_tensor, tuple):
                dev = output_tensor[0].device
                return (output_tensor[0] + alpha_t * v_vector.to(dev),) + output_tensor[1:]
            dev = output_tensor.device
            return output_tensor + alpha_t * v_vector.to(dev)
        return output_tensor
    return hook_fn

print("🚀 Running Model Inference across N_test = 500 prompts with v_perm...")
generated_texts, ref_answers, hal_answers = [], [], []

for item in tqdm(test_data, desc="Inference 500"):
    q_text = item['question']
    prompt = f"<|im_start|>user\n{q_text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]
    
    hook_h = target_module.register_forward_hook(make_decay_hook(v_perm))
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    hook_h.remove()
    
    gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    generated_texts.append(gen_text)
    ref_answers.append(item.get('right_answer', item.get('positive_answer')))
    hal_answers.append(item['hallucinated_answer'])

print("📊 Computing BERTScore Evaluation...")
bs_ref = bertscore.compute(predictions=generated_texts, references=ref_answers, model_type="bert-base-multilingual-cased")['f1']
bs_hal = bertscore.compute(predictions=generated_texts, references=hal_answers, model_type="bert-base-multilingual-cased")['f1']

preferred_count = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h)
refpref_pct = (preferred_count / len(test_data)) * 100.0
mean_bs_f1 = np.mean(bs_ref)

print(f"====================================================================")
print(f"EMPIRICAL 50% BALANCED LABEL PERMUTATION RESULTS:")
print(f"  - RefPref Accuracy: {refpref_pct:.2f}% ({preferred_count} / {len(test_data)})")
print(f"  - Reference BERTScore F1: {mean_bs_f1:.4f}")
print(f"====================================================================")

results_data = {
    "experiment": "Exp09_Full_50pct_Label_Permutation_Kaggle",
    "n_flipped": n_flipped,
    "n_train": n_train,
    "flip_pct": float(n_flipped / n_train * 100.0),
    "n_test": len(test_data),
    "refpref_pct": refpref_pct,
    "preferred_count": preferred_count,
    "mean_bertscore_f1": float(mean_bs_f1)
}

with open("exp09_50pct_label_permutation_results.json", "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2)

print("💾 Results saved to exp09_50pct_label_permutation_results.json")
